## Main

In [1]:
#| default_exp mass.massmain_module

In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
#| export

from anthropmass.mass.com_calculation_module import*
from anthropmass.mass.com_calculation_module import *
from anthropmass.mass.volume_calculation_module import *
from anthropmass.mass.plot_body_module import *
from anthropmass.mass.inertial_calc_module import *
from anthropmass.mass.massclauser_module import *
from anthropmass.mass.measurements_heights_module import get_measurements, get_heights


In [ ]:
#| export
def main(Ansur, inputweight, inputheight, gender):
    
    
    import pandas as pd
    # Define the Ansur data array
    #Ansur = [10027, 266, 1467, 337, 222, 1347, 253, 202, 401, 369, 274, 493, 71, 319, 291, 142,
    #         979, 240, 882, 619, 509, 373, 1535, 291, 1074, 259, 1292, 877, 607, 351, 36, 71, 19,
     #        247, 802, 101, 273, 349, 299, 575, 477, 1136, 90, 214, 193, 150, 583, 206, 326, 70,
      #       332, 366, 1071, 685, 422, 441, 502, 560, 500, 77, 391, 118, 400, 436, 1447, 113, 437,
       ##     919, 1700, 501, 329, 933, 240, 440, 1054, 815, 175, 853, 'Male', '4-Oct-10', 'Fort Hood',
         #    'Regular Army', 'Combat Arms', '19D', 'North Dakota', 1, 1, 41, 71, 180, 'Right hand']

    # Compute measurements, heights, volumes, and COM points
    measurements = get_measurements(Ansur, inputheight)
    heights = get_heights(Ansur, inputheight)
    volumes = get_volumes(Ansur, inputheight)
    pointsJC, com = get_joint_and_com_points(Ansur, inputheight, gender)
    weights = clauser(Ansur.iloc[0], inputweight, inputheight)
    weightsZat = zatsiorsky(inputweight, inputheight)
    inertia = calculate_inertia(Ansur, inputweight, inputheight)

    




      # Limb segments
    densities = {
      "densHand":      weights["mH"]   / volumes["vH"],
      "densForearm":   weights["mLA"]  / volumes["vLA"],
      "densUpperArm":  weights["mUA"]  / volumes["vUA"],
      "densFoot":      weights["mF"]   / volumes["vF"],
      "densShank":     weights["mS"]   / volumes["vS"],
      "densThigh":     weights["mTh"]  / volumes["vT"],
      "densHead":      weights["mHe"]  / volumes["vHe"],
      "densTrunk":     weights["mTr"]  / (volumes["vUT"] + volumes["vMT"] + volumes["vLT"]),
      "densBodyTotal": weights["mTOT"] / volumes["vTOT"]
     }


    heights = heights.iloc[0]



    #Calculate weight from regression model:


    # Print results from volume estimation


    
    print("RESULTS:")

    #print("Actual Weight (kg):", inputweight)
    # Actual height is given in the Ansur list (index 75) in meters
    actual_height = inputheight
    print("Estimated Height (m):", heights["TotalH"])
    # or another computed value
    print("Actual Height (m):", actual_height/1000)

    print("Total Estimated Weight (kg) using Clauser regression model",weights['mTOT'] )
    print("Total Estimated Weight (kg) using Zatriosky regression model",weightsZat['mTOTzat'] )



    segments = [
        'Head', 'Upper Trunk', 'Middle Trunk', 'Lower Trunk', 'Upper Arm',
        'Lower Arm', 'Hand', 'Thigh', 'Shank', 'Foot'
    ]

    # Updated coordinate frame:
    # X = Width (set to 0 as placeholder), Y = Depth (from original x), Z = Height (from original y)
    positions_df = pd.DataFrame(index=['X', 'Y', 'Z'], columns=segments)

    positions_df.loc[:, 'Head'] = [0, round(com['HeMC'][0], 3), round(com['HeMC'][1], 3)]
    positions_df.loc[:, 'Upper Trunk'] = [0, round(com['UTMC'][0], 3), round(com['UTMC'][1], 3)]
    positions_df.loc[:, 'Middle Trunk'] = [0, round(com['MTMC'][0], 3), round(com['MTMC'][1], 3)]
    positions_df.loc[:, 'Lower Trunk'] = [0, round(com['LTMC'][0], 3), round(com['LTMC'][1], 3)]

    positions_df.loc[:, 'Upper Arm'] = [0, round(com['RUAMC'][0], 3), round(com['RUAMC'][1], 3)]
    positions_df.loc[:, 'Lower Arm'] = [0, round(com['RLAMC'][0], 3), round(com['RLAMC'][1], 3)]
    positions_df.loc[:, 'Hand'] = [0, round(com['RHMC'][0], 3), round(com['RHMC'][1], 3)]
    positions_df.loc[:, 'Thigh'] = [0, round(com['RTHMC'][0], 3), round(com['RTHMC'][1], 3)]
    positions_df.loc[:, 'Shank'] = [0, round(com['RSHMC'][0], 3), round(com['RSHMC'][1], 3)]
    positions_df.loc[:, 'Foot'] = [0, round(com['RFMC'][0], 3), round(com['RFMC'][1], 3)]

    print("\nPositions of Body Segments (Custom Cartesian Mapping)\n")
    print(positions_df.to_string())

    # Keep only the joint names without the "Right" prefix
    joints = ["Shoulder", "Elbow", "Wrist", "Hip", "Knee", "Ankle"]

    # Create the DataFrame
    jc_positions_df = pd.DataFrame(index=['X', 'Y', 'Z'], columns=joints)

    # Fill the table with absolute Y values (Depth)
    for joint in joints:
        coords = pointsJC[joint]
        y_depth = round(abs(coords[0]), 3)   # take absolute value of Y (depth)
        z_height = round(coords[1], 3)
        jc_positions_df.loc[:, joint] = [0, y_depth, z_height]

    # Display the table
    print("\nJoint Center Positions\n")
    print(jc_positions_df.to_string())


    print("...................")

    # Mass segments with full trunk detail
    mass_segments = [
        "Head", "Trunk Total", "Upper Trunk", "Middle Trunk", "Lower Trunk",
        "Upper Arm", "Forearm", "Hand", "Thigh", "Shank", "Foot"
    ]

    # Create mass comparison table
    mass_df = pd.DataFrame(index=["Clauser", "Zatsiorsky"], columns=mass_segments)

    # --- Clauser Values ---
    mass_df.loc["Clauser", "Head"] = round(weights["mHe"], 3)
    mass_df.loc["Clauser", "Trunk Total"] = round(weights["mTr"], 3)
    mass_df.loc["Clauser", "Upper Trunk"] = "---"
    mass_df.loc["Clauser", "Middle Trunk"] = "---"
    mass_df.loc["Clauser", "Lower Trunk"] = "---"
    mass_df.loc["Clauser", "Upper Arm"] = round(weights["mUA"], 3)
    mass_df.loc["Clauser", "Forearm"] = round(weights["mLA"], 3)
    mass_df.loc["Clauser", "Hand"] = round(weights["mH"], 3)
    mass_df.loc["Clauser", "Thigh"] = round(weights["mTh"], 3)
    mass_df.loc["Clauser", "Shank"] = round(weights["mS"], 3)
    mass_df.loc["Clauser", "Foot"] = round(weights["mF"], 3)

    # --- Zatsiorsky Values ---
    trunk_total_zat = (
        weightsZat["mUppTrzat"] +
        weightsZat["mMidTrzat"] +
        weightsZat["mLowTrzat"]
    )

    mass_df.loc["Zatsiorsky", "Head"] = round(weightsZat["mHezat"], 3)
    mass_df.loc["Zatsiorsky", "Trunk Total"] = round(trunk_total_zat, 3)
    mass_df.loc["Zatsiorsky", "Upper Trunk"] = round(weightsZat["mUppTrzat"], 3)
    mass_df.loc["Zatsiorsky", "Middle Trunk"] = round(weightsZat["mMidTrzat"], 3)
    mass_df.loc["Zatsiorsky", "Lower Trunk"] = round(weightsZat["mLowTrzat"], 3)
    mass_df.loc["Zatsiorsky", "Upper Arm"] = round(weightsZat["mUAzat"], 3)
    mass_df.loc["Zatsiorsky", "Forearm"] = round(weightsZat["mLAzat"], 3)
    mass_df.loc["Zatsiorsky", "Hand"] = round(weightsZat["mHzat"], 3)
    mass_df.loc["Zatsiorsky", "Thigh"] = round(weightsZat["mThzat"], 3)
    mass_df.loc["Zatsiorsky", "Shank"] = round(weightsZat["mSzat"], 3)
    mass_df.loc["Zatsiorsky", "Foot"] = round(weightsZat["mFzat"], 3)

    # --- Display the table ---
    print("\nSegment Mass Comparison (kg) — Detailed Trunk View\n")
    print(mass_df.to_string())




#INERTIA:




    print(f"{'Upper Trunk':<12} {inertia['Ixx_UT']:>12.4f} {inertia['Iyy_UT']:>12.4f} {inertia['Izz_UT']:>12.4f}")
    print(f"{'Middle Trunk':<12} {inertia['Ixx_MT']:>12.4f} {inertia['Iyy_MT']:>12.4f} {inertia['Izz_MT']:>12.4f}")
    print(f"{'Lower Trunk':<12} {inertia['Ixx_LT']:>12.4f} {inertia['Iyy_LT']:>12.4f} {inertia['Izz_LT']:>12.4f}")

    print(f"{'Thigh':<12} {inertia['Ixx_T']:>12.4f} {inertia['Iyy_T']:>12.4f} {inertia['Izz_T']:>12.4f}")
    print(f"{'Shank':<12} {inertia['Ixx_S']:>12.4f} {inertia['Iyy_S']:>12.4f} {inertia['Izz_S']:>12.4f}")
    print(f"{'Foot':<12} {inertia['Ixx_F']:>12.4f} {inertia['Iyy_F']:>12.4f} {inertia['Izz_F']:>12.4f}")

    print(f"{'Upper Arm':<12} {inertia['Ixx_UA']:>12.4f} {inertia['Iyy_UA']:>12.4f} {inertia['Izz_UA']:>12.4f}")
    print(f"{'Lower Arm':<12} {inertia['Ixx_LA']:>12.4f} {inertia['Iyy_LA']:>12.4f} {inertia['Izz_LA']:>12.4f}")
    print(f"{'Hand':<12} {inertia['Ixx_Ha']:>12.4f} {inertia['Iyy_Ha']:>12.4f} {inertia['Izz_Ha']:>12.4f}")
    print(f"{'Head':<12} {inertia['Ixx_H']:>12.4f} {inertia['Iyy_H']:>12.4f} {inertia['Izz_H']:>12.4f}")


    print(inertia)





















    # Draw the plot
    plot_body(Ansur, inputheight, gender)

    

    
    


In [19]:
# from anthropmass.prediction_module import *
# import pandas as pd
# train=pd.read_csv('../data/processed/ANSURIInormalizedtrain.csv')
# y_train=train.iloc[:,1:94].drop('weightkg',axis=1).drop('stature',axis=1)
# variables = y_train.columns[:]
import nbdev; nbdev.nbdev_export()


In [6]:
def make_a_man(weight, height, gender):
    df = make_predictions('xgboost',variables, weight, height, gender)
    main(df, weight, height, gender)
    

In [7]:
# make_a_man(80,1800,1)

In [8]:
# make_a_man(40,1600,1)